In [10]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim

In [11]:
# Step 1: Load datasets
normal_dataset = pd.read_csv(r"Data\PMSM Data\High Speed Config.csv")
#short_circuit_dataset = pd.read_csv(r"Data\PMSM Data\closed circuit config pmsm.csv")
#open_circuit_dataset = pd.read_csv(r"Data\PMSM Data\open circuit config pmsm.csv")
#ground_fault_dataset = pd.read_csv(r"Data\PMSM Data\Ground circuit config pmsm.csv")



In [34]:
# Step 2: Combine datasets row by row
combined_dataset = pd.concat([normal_dataset, open_circuit_dataset, short_circuit_dataset, ground_fault_dataset], axis=0, ignore_index=True)

# Step 3: Assign labels
combined_dataset['fault_type'] = np.nan
combined_dataset.loc[normal_dataset.index, 'fault_type'] = 'Normal'
combined_dataset.loc[open_circuit_dataset.index, 'fault_type'] = 'Open Circuit'
combined_dataset.loc[short_circuit_dataset.index, 'fault_type'] = 'Short Circuit'
combined_dataset.loc[ground_fault_dataset.index, 'fault_type'] = 'Ground Fault'


C:\Users\Derrick Baalaboore\AppData\Local\Temp\ipykernel_32448\4220760158.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'Normal' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  combined_dataset.loc[normal_dataset.index, 'fault_type'] = 'Normal'


In [35]:
# Step 4: Data Preprocessing
# Remove rows with missing values
combined_dataset = combined_dataset.fillna(combined_dataset.mean())
print(combined_dataset.shape)


# Step 4: Separate features and target variable
#X = combined_dataset.drop(columns=['fault_type'])
#y = combined_dataset['fault_type']

# Step 5: Normalization (only for feature columns)
#scaler = StandardScaler()
#X_scaled = scaler.fit_transform(X)

In [5]:
# Step 3: Split data into training and validation sets
def split_data(features, labels):
    X_train, X_val, y_train, y_val = train_test_split(features, labels, test_size=0.2, random_state=42)
    return X_train, X_val, y_train, y_val

X_train_sc, X_val_sc, y_train_sc, y_val_sc = split_data(features_sc, labels_sc)
X_train_oc, X_val_oc, y_train_oc, y_val_oc = split_data(features_oc, labels_oc)
X_train_gf, X_val_gf, y_train_gf, y_val_gf = split_data(features_gf, labels_gf)


In [6]:
# Step 4: Define ANN model
class ANN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(ANN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out


In [7]:
# Step 5: Train the model
def train_model(model, criterion, optimizer, X_train, y_train, X_val, y_val, epochs=100, batch_size=64):
    train_losses = []
    val_losses = []
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
        
        # Validation loss
        model.eval()
        outputs_val = model(X_val)
        val_loss = criterion(outputs_val, y_val)
        val_losses.append(val_loss.item())
        
        if epoch % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Train Loss: {loss.item()}, Val Loss: {val_loss.item()}")
    
    return train_losses, val_losses

input_size = X_train_sc.shape[1]
hidden_size = 128
num_classes = len(np.unique(labels_sc))
model = ANN(input_size, hidden_size, num_classes)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

train_losses_sc, val_losses_sc = train_model(model, criterion, optimizer, torch.Tensor(X_train_sc.values), 
                                             torch.LongTensor(y_train_sc.values), torch.Tensor(X_val_sc.values), 
                                             torch.LongTensor(y_val_sc.values))


Epoch 1/100, Train Loss: nan, Val Loss: nan
Epoch 11/100, Train Loss: nan, Val Loss: nan
Epoch 21/100, Train Loss: nan, Val Loss: nan
Epoch 31/100, Train Loss: nan, Val Loss: nan
Epoch 41/100, Train Loss: nan, Val Loss: nan


KeyboardInterrupt: 

In [9]:
# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)  # Ensure labels are of type long
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.long)  # Ensure labels are of type long


In [9]:
# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)  # Use dtype=torch.long for multiclass classification
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.long)


# Create PyTorch DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)



In [11]:
# Define model architecture
class ANN(nn.Module):
    def __init__(self, input_size, num_classes):
        super(ANN, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 16)
        self.fc4 = nn.Linear(16, num_classes)  # Adjust output size for the number of classes

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.fc4(x)
        return x

# Instantiate model
input_size = X_train.shape[1]
num_classes = len(label_encoder.classes_)
model = ANN(input_size, num_classes)

In [12]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Use CrossEntropyLoss for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [15]:
# Create PyTorch DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

In [17]:
# Train model
num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss}")


Epoch 1/50, Loss: nan


KeyboardInterrupt: 